In [17]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,       # Vocabulary size
    "context_length": 256,    # Context length
    "emb_dim": 768,            # Embedding dimension
    "n_heads": 12,             # Number of attention heads
    "n_layers": 12,            # Number of layers
    "drop_rate": 0.1,          # Dropout rate
    "qkv_bias": False          # Query-Key-Value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

In [2]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
import numpy as np

## from Chapter 3

In [8]:
# from chapter 3

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # Ensure that the output dimension can be evenly split across heads
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        # Total output dimension of attention layer
        self.d_out = d_out

        # Number of attention heads
        self.num_heads = num_heads

        # Dimension handled by each head
        self.head_dim = d_out // num_heads

        # Linear projections to produce Query, Key, and Value vectors
        # Each token embedding is projected into d_out dimension
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Final linear layer to mix the outputs of all heads
        self.out_proj = nn.Linear(d_out, d_out)

        # Dropout applied to attention weights (regularization)
        self.dropout = nn.Dropout(dropout)

        # Causal mask (upper triangular matrix)
        # Prevents tokens from attending to future tokens in autoregressive models
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):

        # x shape: (batch_size, num_tokens, input_dimension)
        b, num_tokens, d_in = x.shape

        # Project input embeddings into query, key, and value vectors
        # Shape after projection: (batch_size, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split each projection into multiple heads
        # New shape: (batch_size, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Move the head dimension before the token dimension
        # New shape: (batch_size, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        queries = queries.transpose(1, 2)

        # Compute attention scores using scaled dot-product attention
        # scores shape: (batch_size, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # Apply causal mask so tokens cannot attend to future tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Scale scores by sqrt(head_dim) for numerical stability
        # Then convert scores to probabilities with softmax
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # Apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)

        # Compute weighted sum of value vectors
        # Result shape: (batch_size, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Merge all heads back together
        # Shape becomes: (batch_size, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        # Final linear projection after concatenating heads
        context_vec = self.out_proj(context_vec)

        return context_vec

## from Chapter 4

In [6]:
# from chapter 4

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        # Small constant added to variance for numerical stability
        self.eps = 1e-5

        # Learnable scaling parameter (gamma in LayerNorm literature)
        # One value per embedding dimension
        self.scale = nn.Parameter(torch.ones(emb_dim))

        # Learnable shift parameter (beta)
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # Compute mean across embedding dimension
        # Shape: (batch, seq_len, 1)
        mean = x.mean(dim=-1, keepdim=True)

        # Compute variance across embedding dimension
        # unbiased=False matches typical LayerNorm implementation
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalize input: (x - mean) / sqrt(var + eps)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        # Apply learnable scale and shift
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Gaussian Error Linear Unit activation
        # Smooth alternative to ReLU used in GPT/BERT
        # This is the tanh approximation of GELU
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi))
                * (x + 0.044715 * torch.pow(x, 3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Position-wise feed-forward network
        # Expands embedding dimension then projects back
        self.layers = nn.Sequential(

            # First linear layer expands dimension (typically 4x)
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),

            # Non-linear activation
            GELU(),

            # Project back to original embedding size
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        # Applies feed-forward network to each token independently
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Multi-head self-attention layer
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )

        # Position-wise feed-forward network
        self.ff = FeedForward(cfg)

        # Layer normalization before attention
        self.norm1 = LayerNorm(cfg["emb_dim"])

        # Layer normalization before feed-forward
        self.norm2 = LayerNorm(cfg["emb_dim"])

        # Dropout applied to residual connections
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):

        # ---- Self-Attention Block ----

        # Save residual (shortcut) connection
        shortcut = x

        # Apply layer normalization
        x = self.norm1(x)

        # Apply multi-head self-attention
        x = self.att(x)

        # Apply dropout
        x = self.drop_shortcut(x)

        # Add residual connection
        x = x + shortcut


        # ---- Feed-Forward Block ----

        # Save residual connection again
        shortcut = x

        # Apply second normalization
        x = self.norm2(x)

        # Apply feed-forward network
        x = self.ff(x)

        # Apply dropout
        x = self.drop_shortcut(x)

        # Add residual connection
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Token embedding layer
        # Converts token IDs into embedding vectors
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # Positional embedding layer
        # Adds information about token position in the sequence
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # Dropout applied to embeddings
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Stack of Transformer blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Final layer normalization
        self.final_norm = LayerNorm(cfg["emb_dim"])

        # Output projection layer
        # Maps embeddings back to vocabulary logits
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):

        # Input shape: (batch_size, sequence_length)
        batch_size, seq_len = in_idx.shape

        # Convert token IDs into embeddings
        # Shape: (batch, seq_len, emb_dim)
        tok_embeds = self.tok_emb(in_idx)

        # Generate position indices (0...seq_len-1)
        # Then convert them to positional embeddings
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        # Combine token and positional embeddings
        x = tok_embeds + pos_embeds

        # Apply dropout
        x = self.drop_emb(x)

        # Pass through transformer blocks
        x = self.trf_blocks(x)

        # Final normalization
        x = self.final_norm(x)

        # Project embeddings to vocabulary logits
        logits = self.out_head(x)

        return logits

## Helpers

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("cuda" if torch.cuda.is_available() else "cpu")

cuda


In [10]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)   
    return encoded_tensor
    
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)               
    return tokenizer.decode(flat.tolist())

In [11]:
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves"
txt2 = "I really like"

batch.append(text_to_token_ids(txt1, tokenizer))
batch.append(text_to_token_ids(txt2, tokenizer))
inp = torch.cat(batch, dim=0)

In [3]:
def generate(model, idx, max_new_tokens, context_size,
     temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):           
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        
        if top_k is not None:               
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )
        if temperature > 0.0:                 
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:   
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if idx_next == eos_id:             
            break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

## Test Model

In [20]:
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("models/model.pth", map_location=device))
model.to(device)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [21]:
torch.manual_seed(123)
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.6
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as terr what one can rather a cheap genius--I looked with equanim
